# 🤖 Colab Remote Control Agent

Run all cells in this notebook. It will:
1. Prompt you for a secret API token and your ngrok auth token.
2. Install `flask` and `pyngrok`.
3. Start a tiny REST server inside Colab and expose it via ngrok.

Copy the **Public URL** and use the same API secret in your local MCP server environment variables (`COLAB_NGROK_URL` and `COLAB_API_SECRET`).

In [ ]:
# ✅ Configuration (prompts for secrets so they are not stored in the notebook)
import getpass
import os

agent_secret = getpass.getpass('Enter a strong API secret for the MCP agent: ')
if not agent_secret:
    raise ValueError('API secret is required')
os.environ['COLAB_AGENT_SECRET'] = agent_secret

ngrok_token = getpass.getpass('Enter your ngrok auth token (press Enter if already configured globally): ')
if ngrok_token:
    os.environ['COLAB_NGROK_TOKEN'] = ngrok_token

In [ ]:
# 📦 Install dependencies
!pip install -q flask pyngrok

In [ ]:
# 🚀 Start the agent
import os
import io
import subprocess
import threading
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

API_SECRET = os.environ.get('COLAB_AGENT_SECRET', '')
PORT = 5000
MAX_OUTPUT_CHARS = 100_000
MAX_FILE_SIZE = 5 * 1024 * 1024  # 5 MB


def _check_auth(request):
    auth = request.headers.get('Authorization', '')
    return auth.startswith('Bearer ') and auth.split(' ', 1)[1] == API_SECRET


def _capture_exec(code, globals_dict):
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()
    result = {'status': 'success', 'output': '', 'error': ''}
    try:
        with redirect_stdout(stdout_capture), redirect_stderr(stderr_capture):
            exec(code, globals_dict)
        result['output'] = stdout_capture.getvalue()
        if stderr_capture.getvalue():
            result['output'] += '\n[stderr]\n' + stderr_capture.getvalue()
    except Exception as e:
        result['status'] = 'error'
        result['error'] = f'{type(e).__name__}: {e}'

    if len(result['output']) > MAX_OUTPUT_CHARS:
        result['output'] = result['output'][:MAX_OUTPUT_CHARS] + '\n... [truncated]'
    return result


@app.route('/health', methods=['GET', 'POST'])
def health():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401
    return jsonify({'status': 'ok'})


@app.route('/execute', methods=['POST'])
def execute():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    code = data.get('code', '')
    command = data.get('command', '')

    if code:
        return jsonify(_capture_exec(code, globals()))
    elif command:
        try:
            proc = subprocess.run(
                command, shell=True, capture_output=True, text=True, timeout=120
            )
            output = proc.stdout
            if proc.stderr:
                output += '\n[stderr]\n' + proc.stderr
            if len(output) > MAX_OUTPUT_CHARS:
                output = output[:MAX_OUTPUT_CHARS] + '\n... [truncated]'
            return jsonify({'status': 'success', 'output': output})
        except Exception as e:
            return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})
    else:
        return jsonify({'status': 'error', 'error': 'No code or command provided'}), 400


@app.route('/read_file', methods=['POST'])
def read_file():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    path = data.get('path', '')
    if not path:
        return jsonify({'status': 'error', 'error': 'Path is required'}), 400

    try:
        file_path = Path(path)
        if not file_path.exists():
            return jsonify({'status': 'error', 'error': 'File not found'}), 404
        if file_path.stat().st_size > MAX_FILE_SIZE:
            return jsonify({'status': 'error', 'error': 'File too large to read'}), 400
        content = file_path.read_text(encoding='utf-8')
        return jsonify({'status': 'success', 'content': content})
    except Exception as e:
        return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})


@app.route('/write_file', methods=['POST'])
def write_file():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    path = data.get('path', '')
    content = data.get('content', '')
    if not path:
        return jsonify({'status': 'error', 'error': 'Path is required'}), 400

    try:
        file_path = Path(path)
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content, encoding='utf-8')
        return jsonify({'status': 'success', 'path': str(file_path)})
    except Exception as e:
        return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})


@app.route('/list_files', methods=['POST'])
def list_files():
    if not _check_auth(request):
        return jsonify({'status': 'error', 'error': 'Unauthorized'}), 401

    data = request.get_json(force=True, silent=True) or {}
    path = data.get('path', '')
    if not path:
        return jsonify({'status': 'error', 'error': 'Path is required'}), 400

    try:
        target = Path(path)
        if not target.exists():
            return jsonify({'status': 'error', 'error': 'Path not found'}), 404
        if not target.is_dir():
            return jsonify({'status': 'error', 'error': 'Path is not a directory'}), 400
        files = []
        for item in target.iterdir():
            label = item.name + '/' if item.is_dir() else item.name
            files.append(label)
        return jsonify({'status': 'success', 'files': files})
    except Exception as e:
        return jsonify({'status': 'error', 'error': f'{type(e).__name__}: {e}'})


def run_flask():
    app.run(host='0.0.0.0', port=PORT, use_reloader=False)


ngrok_token = os.environ.get('COLAB_NGROK_TOKEN')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

try:
    public_url = ngrok.connect(PORT)
    print('Colab agent is running.')
    print(f'Public URL: {public_url}')
    print(f"API Secret: {API_SECRET[:4]}{'*' * max(0, len(API_SECRET) - 4)}")
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
except Exception as exc:
    print(f'Failed to start ngrok tunnel: {exc}')
    raise